# VBN Session Preview

This notebook demonstrates how to load and preview data from a Visual Behavior Neuropixels session.

Before running this notebook, ensure you have:
1. Installed the package: `pip install -e .`
2. Downloaded metadata: `python scripts/download_metadata.py`
3. Downloaded the session: `python scripts/download_session.py 1055240613`

In [ ]:
# Cell 1: Setup and imports
from vbn.config import load_config, get_cache_dir, get_outputs_dir
from vbn.cache import get_cache, get_sessions_table
from vbn.io import load_session, get_eye_tracking, get_stimulus_presentations, get_running_speed, summarize_session
from vbn.video import discover_videos
from vbn.pose import PoseOutput, validate_pose_schema

import matplotlib.pyplot as plt
import pandas as pd

SESSION_ID = 1055240613

print(f"Cache directory: {get_cache_dir()}")
print(f"Outputs directory: {get_outputs_dir()}")

In [ ]:
# Cell 2: Initialize cache and load session
cache = get_cache()
session = load_session(cache, SESSION_ID)

# Print summary
summary = summarize_session(session)
print(f"\nSession {SESSION_ID} Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
# Cell 3: Check for video files
videos = discover_videos(get_cache_dir(), SESSION_ID)
print(f"Video files found: {len(videos)}")

if videos:
    for v in videos:
        print(f"  - {v['path'].name} ({v['camera_type_guess']}, {v['size_mb']:.1f} MB)")
else:
    print("No standalone video files found.")
    print("The VBN public dataset includes processed eye tracking but not raw videos.")

In [ ]:
# Cell 4: Load and visualize eye tracking data
eye_df = get_eye_tracking(session)

if eye_df is not None:
    print(f"Eye tracking data: {len(eye_df):,} samples")
    print(f"Columns: {list(eye_df.columns)}")
    
    # Plot first 5 minutes
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    
    # Limit to first 5 minutes
    t_max = min(300, eye_df.index.max())
    subset = eye_df[eye_df.index <= t_max]
    
    axes[0].plot(subset.index, subset['pupil_center_x'], 'b-', linewidth=0.5)
    axes[0].set_ylabel('Pupil X')
    axes[0].set_title('Eye Tracking (first 5 minutes)')
    
    axes[1].plot(subset.index, subset['pupil_center_y'], 'g-', linewidth=0.5)
    axes[1].set_ylabel('Pupil Y')
    
    axes[2].plot(subset.index, subset['pupil_area'], 'r-', linewidth=0.5)
    axes[2].set_ylabel('Pupil Area')
    axes[2].set_xlabel('Time (s)')
    
    plt.tight_layout()
    plt.show()
else:
    print("No eye tracking data available")

In [ ]:
# Cell 5: Running speed
running_df = get_running_speed(session)

if len(running_df) > 0:
    print(f"Running speed data: {len(running_df):,} samples")
    
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(running_df['timestamps'], running_df['speed'], 'k-', linewidth=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Running Speed (cm/s)')
    ax.set_title('Running Speed')
    ax.set_xlim(0, 300)  # First 5 minutes
    plt.tight_layout()
    plt.show()
else:
    print("No running speed data available")

In [ ]:
# Cell 6: Stimulus presentations
stim_df = get_stimulus_presentations(session)

print(f"Stimulus presentations: {len(stim_df)}")
print(f"\nColumns: {list(stim_df.columns)[:10]}...")  # First 10 columns

# Show stimulus types
if 'image_name' in stim_df.columns:
    print(f"\nImage names:")
    print(stim_df['image_name'].value_counts())

In [ ]:
# Cell 7: Pupil position scatter
if eye_df is not None:
    # Sample for plotting
    sample = eye_df.sample(min(10000, len(eye_df)))
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    scatter = ax.scatter(
        sample['pupil_center_x'], 
        sample['pupil_center_y'],
        c=sample.index,
        cmap='viridis',
        s=1,
        alpha=0.3
    )
    
    ax.set_xlabel('Pupil X')
    ax.set_ylabel('Pupil Y')
    ax.set_title('Pupil Position Distribution')
    plt.colorbar(scatter, label='Time (s)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 8: Next steps
print("Next Steps:")
print("="*50)
print("")
print("1. Generate video preview:")
print(f"   python scripts/preview_video.py {SESSION_ID}")
print("")
print("2. Export frames for labeling:")
print(f"   python scripts/export_frames.py {SESSION_ID} --n-frames 100")
print("")
print("3. Sample frames for labeling:")
print(f"   python scripts/sample_label_frames.py {SESSION_ID} --n-samples 50")
print("")
print("4. After labeling, train pose model:")
print("   python scripts/train_sleap.py path/to/project.slp")
print("   python scripts/train_dlc.py path/to/config.yaml")